# 01. LangGraph Functional API: @entrypoint와 @task

> 그래프 빌더 API가 부담스럽다면 `@entrypoint` · `@task` 데코레이터만으로도 같은 LangGraph 기능을 쓸 수 있어요. 함수형 스타일이 잘 어울리는 케이스를 골라 비교해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `@entrypoint`와 `@task` 데코레이터를 사용해 워크플로우를 함수형으로 정의할 수 있어요
2. Graph API와 Functional API의 차이를 설명하고 적절한 상황에 각 API를 선택할 수 있어요
3. `previous`, `store`, `writer`, `config` Injectable 파라미터를 활용할 수 있어요
4. `entrypoint.final`로 반환값과 저장값을 분리하고, 병렬 실행으로 성능을 높일 수 있어요
5. `interrupt`와 `Command(resume=)`을 Functional API에서 사용해 Human-in-the-Loop를 구현할 수 있어요

## 사전 지식

- Part 02: StateGraph, Node, Edge, START/END로 그래프 구성 방법
- Part 02: InMemorySaver, thread_id를 이용한 체크포인팅
- Part 03: LangGraph의 "Thinking" 방식, 워크플로우 vs 에이전트 구분, 5가지 사고 도구(Constrain·Inform·Verify·Correct·HITL)

## 이전 챕터와의 연결 — 이번 챕터가 5가지 도구의 어디를 심화하나요?

Part 03/03에서 배운 5가지 사고 도구는 03~13장 전체의 공통 언어예요. Part 04(LangGraph Advanced)는 그중에서도 주로 **Correct** 와 **HITL** 을 깊이 다져요.

| Part 04 레슨 | 주로 강화하는 사고 도구 | 이유 |
|-------------|------------------------|------|
| **01 Functional API** (이 노트북) | Correct | `@task` 결과를 체크포인트에 기록해 실패 지점부터 자동 재개 — 실패를 "다음 호출에 정보를 더해" 복구하는 원리의 기반 |
| **02 Subgraphs** | Constrain + Inform | 하위 그래프로 책임을 나누면 컨텍스트와 도구 권한을 계층적으로 제한 |
| **03 Branching-Parallel** | Constrain | Superstep 트랜잭션과 `defer=True`로 동시 실행 순서·병합을 안전하게 통제 |
| **04 Durable Execution** | Correct | `RetryPolicy` + 내구성 실행으로 네트워크·리소스 오류를 자동 복구 |
| **05 DeleteMessages** / **06 Conversation-Summary** | Inform | 긴 맥락에서 "지금 꼭 보여줄 것만" 유지하는 메시지 수술법 |
| **07 Streaming-Steps** | (관찰성 보조) | 도구 선택은 아니지만 Verify/Correct를 실무에서 작동시키는 관찰 창구 |

이 노트북(01 Functional API)은 단순히 "또 하나의 API"가 아니라, 실패를 받아들이고도 전진하는 **Correct 도구**를 언어 수준에서 구현한 장치예요. Part 03의 `02-Design-Principles.ipynb`에서 StateGraph(Graph API)의 노드·엣지 사고방식을 익혔다면, 이번에는 일반 Python 함수에 데코레이터를 붙이는 것만으로 내구성 있는(durable) 워크플로우를 만드는 법을 살펴봐요.


## Functional API란?

LangGraph는 워크플로우를 정의하는 두 가지 방식을 제공해요.

| 특성 | Graph API (StateGraph) | Functional API (@entrypoint/@task) |
|------|----------------------|------------------------------------|
| 정의 방식 | 노드와 엣지를 명시적으로 선언 | 함수에 데코레이터를 붙여 정의 |
| 제어 흐름 | 그래프 토폴로지로 표현 | 일반 Python 제어 흐름 (if/for/while) |
| 상태 관리 | TypedDict State 스키마 | 함수 입출력 + 자동 체크포인팅 |
| 시각화 | 그래프 다이어그램 지원 | 제한적 (task 레벨만) |
| 학습 곡선 | 중간 (State 스키마 필요) | 낮음 (일반 Python 함수) |
| 복잡한 로직 | 조건 분기가 명시적 | Python 코드로 자연스럽게 표현 |
| 내구성 실행 | 지원 | 지원 |

> 🔑 **핵심 개념**: Functional API의 핵심은 **`@task` 함수 호출이 자동으로 체크포인트에 기록**된다는 거예요. 실패 후 재실행하면 이미 완료된 task는 캐시에서 결과를 가져오고 실패한 시점부터 다시 시작해요. 이것이 "내구성 있는 실행(Durable Execution)"이에요.

### 두 API의 아키텍처 비교

```mermaid
flowchart TB
    subgraph G ["Graph API - StateGraph"]
        direction LR
        GS["START"]:::input --> GN1["node_a"]:::process
        GN1 --> GN2["node_b"]:::process
        GN2 --> GN3["node_c"]:::process
        GN3 --> GE["END"]:::output
    end

    subgraph F ["Functional API - @entrypoint"]
        direction LR
        FE["@entrypoint"]:::input --> FT1["@task A"]:::process
        FT1 --> FT2["@task B"]:::process
        FT2 --> FT3["@task C"]:::process
        FT3 --> FR["return"]:::output
    end

    subgraph CP ["공통: Checkpointer"]
        direction LR
        CS[("체크포인트<br>저장소")]:::storage
    end

    G -.->|체크포인팅| CP
    F -.->|체크포인팅| CP

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```


## 환경 설정


In [ ]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에서 OPENAI_API_KEY 등을 불러와요
from dotenv import load_dotenv
load_dotenv()

# 환경 변수 로드 완료


In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택)
# ---------------------------------------------------
# LangSmith로 워크플로우 실행을 추적하면 디버깅에 도움이 돼요
import os

# LangSmith API 키가 있는 경우 추적을 활성화해요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-Functional-API"

# 설정 완료


## 1. 기본 사용법: @entrypoint와 @task

Functional API의 두 핵심 데코레이터를 소개해요:

- **`@task`**: 개별 작업 단위를 정의해요. 체크포인트에 결과가 기록되며, 재실행 시 캐시에서 결과를 반환해요.
- **`@entrypoint`**: 워크플로우의 진입점이에요. `@task`들을 조합해 전체 흐름을 정의하고, 체크포인터와 연결해요.

### Graph API vs Functional API: 코드 스타일 비교

```python
# Graph API: "그래프를 그리고 로직을 채운다"
builder = StateGraph(State)
builder.add_node("step1", step1_fn)
builder.add_node("step2", step2_fn)
builder.add_edge("step1", "step2")
graph = builder.compile(checkpointer=saver)

# Functional API: "로직을 쓰고 데코레이터가 내구성을 부여한다"
@entrypoint(checkpointer=saver)
def workflow(input):
    result1 = step1_task(input).result()
    result2 = step2_task(result1).result()
    return result2
```


In [ ]:
from langgraph.func import entrypoint, task

from langgraph.checkpoint.memory import InMemorySaver

from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-4o-mini")

    from langchain.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 Functional API 예제
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: @entrypoint → call_model(@task) → return
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 워크플로우 실행하기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 스트리밍 실행
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. 여러 task 조합하기

실제 워크플로우에서는 여러 task를 순서대로 실행하거나 조건에 따라 다르게 실행해요. Functional API에서는 일반 Python 코드로 이를 표현할 수 있어요.


In [ ]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

model2 = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 여러 task를 순서대로 실행하는 워크플로우
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: @entrypoint → generate_outline → write_introduction → create_summary → return
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. 병렬 실행

Functional API의 강점 중 하나는 여러 task를 **병렬로** 실행할 수 있다는 거예요.

### 순차 실행 vs 병렬 실행

```mermaid
flowchart LR
    subgraph SEQ["순차 실행 (느림)"]
        direction LR
        S1["Task A<br/>2초"] --> S2["Task B<br/>2초"] --> S3["Task C<br/>2초"]
    end

    subgraph PAR["병렬 실행 (빠름)"]
        direction TB
        P0["Future 생성"] --> P1["Task A<br/>2초"]
        P0 --> P2["Task B<br/>2초"]
        P0 --> P3["Task C<br/>2초"]
        P1 --> P4[".result() 수집"]
        P2 --> P4
        P3 --> P4
    end

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    class S1,S2,S3,P1,P2,P3 process
    class P0,P4 output
```

| 방식 | 코드 패턴 | 총 소요 시간 (각 2초일 때) |
|------|-----------|------------------------|
| 순차 | `a = task_a(x).result(); b = task_b(x).result()` | 6초 (2+2+2) |
| 병렬 | `futures = [task_a(x), task_b(x), task_c(x)]; results = [f.result() for f in futures]` | 2초 (가장 긴 task 기준) |


In [ ]:
import time
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

model3 = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 병렬 실행 패턴: 먼저 모두 호출하고 나서 결과 수집
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# ---------------------------------------------------
# 순차 실행 vs 병렬 실행 비교
# ---------------------------------------------------
# ============================================================
# TODO: 순차 실행 버전을 작성해보고 병렬 실행과 시간을 비교해요
# 힌트: 아래 코드에서 Future를 먼저 모두 만들지 말고
#       각 task마다 즉시 .result()를 호출하면 순차 실행이 돼요
# 예상 결과: 순차 실행이 병렬 실행보다 약 2~3배 느릴 거예요
# ============================================================


## 4. Injectable 파라미터

Functional API는 `@entrypoint` 함수에 특별한 파라미터를 **자동으로 주입**할 수 있어요. 이를 **Injectable 파라미터**라고 해요.

| 파라미터 | 타입 | 설명 |
|---------|------|------|
| `previous` | 이전 반환값 타입 | 이전 실행의 반환값 (대화 이력 유지에 활용) |
| `store` | `BaseStore` | 크로스 스레드 장기 기억 저장소 |
| `writer` | `StreamWriter` | 커스텀 스트리밍 청크를 내보내는 함수 |
| `config` | `RunnableConfig` | 실행 설정 (thread_id, 사용자 ID 등) |

> 🔑 **핵심 개념**: `previous`는 체크포인터가 있을 때 자동으로 이전 실행의 반환값을 주입해줘요. 대화형 워크플로우에서 이전 대화 내용을 기억하는 데 핵심 역할을 해요. 체크포인터 없이는 사용할 수 없어요.


In [ ]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from typing import Optional

model4 = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: previous 파라미터: 이전 실행 결과를 자동으로 주입받아요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from typing import Optional

model5 = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: config 파라미터: 실행 설정 정보를 주입받아요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. entrypoint.final: 반환값과 저장값 분리

기본적으로 `@entrypoint` 함수의 반환값은 호출자에게 전달되면서 동시에 체크포인트에도 저장돼요. 그런데 때로는 **반환할 값**과 **저장할 값**을 다르게 하고 싶을 때가 있어요.

이럴 때 `entrypoint.final(value=..., save=...)` 을 사용해요.

- `value`: 호출자에게 반환할 값 (예: 사용자에게 보여줄 응답)
- `save`: 체크포인트에 저장할 값 (예: 다음 실행에서 `previous`로 사용할 상태)


In [ ]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from typing import Optional

model6 = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: entrypoint.final: 반환값과 저장값을 분리해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. Human-in-the-Loop: interrupt와 Command

Functional API에서도 Graph API와 동일하게 `interrupt()`로 워크플로우를 일시 중지하고 사람의 입력을 기다릴 수 있어요.

### 실행 흐름

```mermaid
flowchart LR
    A["@entrypoint 시작"] --> B["@task: 초안 생성"]
    B --> C["interrupt()<br/>실행 중지"]
    C -->|"사람이 검토"| D["Command(resume=피드백)"]
    D --> E["@task: 피드백 반영"]
    E --> F["return 최종 결과"]

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef pause fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef human fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A,B,E,F process
    class C pause
    class D human
```

흐름:
1. 워크플로우 실행 중 `interrupt(value)` 호출 -> 실행 중지, `value`를 호출자에게 전달
2. 사람이 검토 후 `Command(resume=...)` 로 재실행
3. `interrupt()` 는 `resume` 값을 반환하며 실행 재개


In [ ]:
from langgraph.func import entrypoint, task
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

model7 = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: interrupt + Command(resume=): 사람의 승인이 필요한 워크플로우
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph.types import Command

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Command(resume=)로 워크플로우 재개하기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. retry_policy와 cache_policy

Functional API는 `@task` 데코레이터에 재시도 정책(`retry_policy`)과 캐시 정책(`cache_policy`)을 설정할 수 있어요.

- **`retry_policy`**: API 호출 실패 시 자동 재시도 설정
- **`cache_policy`**: 동일한 입력에 대해 이전 결과를 재사용


In [ ]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import RetryPolicy
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

model8 = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: retry_policy: 실패 시 자동 재시도 설정
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import time
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.cache.memory import InMemoryCache
from langgraph.types import CachePolicy
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

model9 = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: cache_policy: 동일 입력 재실행 시 캐시에서 반환
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 8. Graph API vs Functional API: 언제 무엇을 쓸까?

두 API를 모두 살펴봤어요. 이제 언제 어떤 API를 선택할지 정리해볼게요.

```mermaid
flowchart TD
    Q1{"그래프 구조를<br>시각화해야 하나요?"}:::input
    Q2{"노드 수가<br>고정되어 있나요?"}:::input
    Q3{"복잡한 Python 로직<br>(if/for/while)이 있나요?"}:::input
    Q4{"기존 Python 코드를<br>마이그레이션하나요?"}:::input

    GA["Graph API<br>(StateGraph)"]:::output
    FA["Functional API<br>(@entrypoint/@task)"]:::output
    BOTH["두 API를 혼합<br>(Functional 안에 Graph)"]:::process

    Q1 -->|예| GA
    Q1 -->|아니오| Q2
    Q2 -->|예| GA
    Q2 -->|아니오| Q3
    Q3 -->|예| FA
    Q3 -->|아니오| Q4
    Q4 -->|예| FA
    Q4 -->|아니오| BOTH

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

### 실전 선택 가이드

| 상황 | 추천 API | 이유 |
|------|---------|------|
| 에이전트 루프 (도구 호출 반복) | Graph API | 반복 구조가 명확하고 시각화가 중요 |
| 동적 단계 수 (루프/조건 복잡) | Functional API | Python 제어 흐름으로 자연스럽게 표현 |
| 기존 Python 함수를 내구성 있게 | Functional API | 데코레이터 추가만으로 전환 가능 |
| 팀 협업, 명확한 워크플로우 문서화 | Graph API | 시각적 그래프로 구조 공유 용이 |
| 복잡한 Human-in-the-Loop | 두 API 모두 지원 | interrupt/Command는 동일하게 작동 |
| 병렬 fan-out/fan-in | Functional API | 코드가 더 직관적 |


In [ ]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import RetryPolicy
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

my_model = init_chat_model("openai:gpt-4o-mini")

# ---------------------------------------------------
# TODO: 자신만의 Functional API 워크플로우 만들기
# ---------------------------------------------------
# ============================================================
# TODO: 아래 요구사항을 만족하는 워크플로우를 작성해보세요
#
# 요구사항:
# 1. 입력: 기술 분야 이름 (예: "블록체인", "양자 컴퓨팅")
# 2. 두 개의 @task를 병렬로 실행:
#    - task 1: 해당 기술의 장점 3가지를 나열
#    - task 2: 해당 기술의 단점 3가지를 나열
# 3. 두 결과를 종합해 최종 평가를 생성하는 세 번째 @task
# 4. retry_policy를 모든 task에 적용
#
# 힌트:
# - 병렬 실행 패턴: futures = [task1(input), task2(input)], results = [f.result() for f in futures]
# - RetryPolicy(max_attempts=2, initial_interval=0.5)
# ============================================================


# TODO: 아래 task들을 완성해보세요

    # TODO: 구현해보세요

    # TODO: 구현해보세요

    # TODO: 구현해보세요

    # TODO: 병렬 실행 패턴을 사용해 두 task를 동시에 실행해보세요


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **`@task`**: 개별 작업 단위를 정의해요. 결과가 체크포인트에 기록되며, 호출 시 Future를 반환하고 `.result()`로 값을 가져와요.
- **`@entrypoint`**: 워크플로우 진입점이에요. `checkpointer`와 연결하고, `@task`들을 일반 Python 코드로 조합해요.
- **병렬 실행**: Future를 먼저 모두 만들고(`[task(input) for ...]`) 나서 결과를 수집(`[f.result() for ...]`)하는 패턴으로 병렬화해요.
- **Injectable 파라미터**: `previous`(이전 반환값), `config`(실행 설정), `store`(장기 저장소), `writer`(커스텀 스트리밍)를 자동 주입받아요.
- **`entrypoint.final`**: `value`(반환값)와 `save`(저장값)를 분리해요.
- **`interrupt` + `Command(resume=)`**: Human-in-the-Loop 패턴을 Functional API에서도 동일하게 사용해요.
- **`RetryPolicy` / `CachePolicy`**: `@task`에 설정해 자동 재시도와 결과 캐싱을 활성화해요.


## 다음 노트북 예고

다음 `02-Subgraphs.ipynb`에서는 **서브그래프(Subgraph)**를 배워요. 복잡한 그래프를 작은 단위로 모듈화하는 **공유 키 패턴**과 **독립 스키마 패턴**, 그리고 3계층 중첩 구조까지 다뤄요. 이번에 배운 `@entrypoint`가 하나의 독립 단위라면, 서브그래프는 **그래프 자체를 재사용 가능한 노드**로 만드는 방법이에요.
